In [ ]:
import pandas as pd

labeling_results = pd.read_csv('data/labeling_full.csv')

# create final codes
labeling_results['final_code'] = labeling_results.apply(
    lambda row: row['annotation'] if pd.notna(row['annotation']) else row['label'], axis=1
)


sorted_results = labeling_results['final_code'].value_counts(ascending=False)
print(len(sorted_results), "unique labels\n")
count_first_order = 0
count_second_order = 0
count_higher_order_single_type = 0
cur_count = -1
operators_for_cur_count = []
for res in sorted_results.index:
    # ensure res as equal length for alignment
    if not res.startswith('Higher-order'):
        if cur_count != sorted_results[res]:
            if cur_count != -1:
                print(", ".join(sorted(operators_for_cur_count)), end="")
                print(" \\\\")
            cur_count = sorted_results[res]
            operators_for_cur_count = []
            print(cur_count, "& ")
        operators_for_cur_count.append(res)
        #print(res, end=", ")
        #print(f"{str(sorted_results[res]).rjust(3)} - {res.ljust(100)}")
        if res.endswith('2nd order)'):
            count_second_order += sorted_results[res]
        elif res.endswith('higher order)'):
            count_higher_order_single_type += sorted_results[res]
        else: 
            count_first_order += sorted_results[res]
print(", ".join(sorted(operators_for_cur_count)), end="")
print(" \\\\")

print("\nHigher-order mutants:")
count_high_order = 0
for res in sorted_results.index:
    # ensure res as equal length for alignment
    if res.startswith('Higher-order'):
        print(f"{sorted_results[res]} & {res} \\\\")
        count_high_order += sorted_results[res]
print(f"\nTotal 1st-order mutants: {count_first_order}")
print(f"\nTotal 2nd-order mutants: {count_second_order}")
print(f"\nTotal higher-order single-type mutants: {count_higher_order_single_type}")
print(f"\nTotal higher-order mutants: {count_high_order}")

# remove 2nd order, higher order from strings
sorted_results.index = sorted_results.index.str.replace('(2nd order)', '')
sorted_results.index = sorted_results.index.str.replace(', 2nd order)', ')')
sorted_results.index = sorted_results.index.str.replace('(higher order)', '')
unique_operators = len(sorted(sorted_results.index[~sorted_results.index.str.startswith('Higher-order')].unique()))
print(f"\nTotal unique 1st-order operators: {unique_operators}")


80 unique labels

38 & 
Replace conditional operator (boundary) \\
23 & 
Replace arithmetic operator \\
19 & 
Insert literal (numeric) \\
17 & 
Replace conditional operator (negation) \\
14 & 
Delete statement (method call), Empty diff \\
9 & 
Replace literal value (numeric) \\
8 & 
Replace variable \\
7 & 
Insert comments, Replace method call with other method, Replace variable (2nd order) \\
6 & 
Delete unary operator (numeric negation), Insert method call, Replace literal value (boolean) \\
5 & 
Delete statement (control flow) \\
4 & 
Delete condition from logical expression, Delete terms from arithmetic expression, Replace conditional operator (boundary, 2nd order) \\
3 & 
Delete statement (assignment), Insert masking, Replace literal value (string), Replace masking, Replace method call with literal, Replace variable with method call \\
2 & 
Delete statement (type cast), Delete unary operator (boolean negation), Insert bit shift, Insert condition into logical expression, Insert ter

In [3]:
eq_mutant_annotations = labeling_results[labeling_results['equivalent']==True]

eqcheck_results = pd.read_csv('data/labeling_equivalent.csv')
# rename unnamed col
eqcheck_results = eqcheck_results.rename(columns={'Unnamed: 0': 'mutant_id'})
eqcheck_results['label'].value_counts().sort_index().sort_values(ascending=False)
full_eq_results = eq_mutant_annotations.merge(eqcheck_results, on='mutant_id', suffixes=('_manual', '_eqcheck'))


sorted_results = full_eq_results[full_eq_results['label_eqcheck']=='equivalent']['final_code'].value_counts(ascending=False)
cur_count = -1
operators_for_cur_count = []
print('Equivalent mutants identified by manual labeling:')
for res in sorted_results.index:
    # ensure res as equal length for alignment
    if cur_count != sorted_results[res]:
        if cur_count != -1:
            print(", ".join(sorted(operators_for_cur_count)), end="")
            print(" \\\\")
        cur_count = sorted_results[res]
        operators_for_cur_count = []
        print(cur_count, "& ")
    operators_for_cur_count.append(res)
print(", ".join(sorted(operators_for_cur_count)), end="")
print(" \\\\")
print()
print('Non-equivalent mutants identified by manual labeling:')
sorted_results = full_eq_results[full_eq_results['label_eqcheck']!='equivalent']['final_code'].value_counts(ascending=False)
cur_count = -1
operators_for_cur_count = []
for res in sorted_results.index:
    # ensure res as equal length for alignment
    if cur_count != sorted_results[res]:
        if cur_count != -1:
            print(", ".join(sorted(operators_for_cur_count)), end="")
            print(" \\\\")
        cur_count = sorted_results[res]
        operators_for_cur_count = []
        print(cur_count, "& ")
    operators_for_cur_count.append(res)
print(", ".join(sorted(operators_for_cur_count)), end="")
print(" \\\\")

Equivalent mutants identified by manual labeling:
8 & 
Replace conditional operator (boundary) \\
4 & 
Insert comments \\
3 & 
Replace method call with literal, Replace variable with method call \\
2 & 
Empty diff, Higher-order: Insert conditional behavior, Higher-order: Replace conditional return of boolean literal with return of boolean expression, Higher-order: Replace pre-increment/decrement with post-increment/decrement, Replace conditional operator (boundary), Insert masking, Insert method call, Replace == with equals \\
1 & 
Delete statement (assignment), Delete statement (method call), Delete statement (type cast, higher order), Higher-order: Insert assignment; Replace method call with variable (2nd order), Higher-order: Replace arithmetic operator, Replace conditional operator (boundary), Replace variable with literal (numeric), Higher-order: Replace declaration, initialization and  arithmetics split into multiple statements with single statement (2nd order), Insert concatenat

In [4]:
full_eq_results.groupby('label_eqcheck')['final_code'].size()

label_eqcheck
equivalent        48
non-equivalent    16
Name: final_code, dtype: int64

In [5]:
all_mutants = pd.read_json('data/mutants_diffs.json', lines=True)
sampled_mutants = pd.read_json('data/mutants_diffs_sampled.json', lines=True)

print('Mutant ratio per model in all mutants:')
display(all_mutants.groupby('attacker')['equivalent'].mean())
print('Mutant ratio per model in sampled mutants:')
display(sampled_mutants.groupby('attacker')['equivalent'].mean())


Mutant ratio per model in all mutants:


attacker
claude-3-5-haiku-latest    0.396739
claude-sonnet-4-0          0.323615
gemini-2.5-flash           0.059625
gemini-2.5-pro             0.064972
gpt-4.1                    0.203343
gpt-4.1-mini               0.201859
Name: equivalent, dtype: float64

Mutant ratio per model in sampled mutants:


attacker
claude-3-5-haiku-latest    0.34
claude-sonnet-4-0          0.44
gemini-2.5-flash           0.06
gemini-2.5-pro             0.04
gpt-4.1                    0.20
gpt-4.1-mini               0.20
Name: equivalent, dtype: float64